# ControlSift — Safe free GPU runner (Kaggle / Colab)

**Security**
- Put your Hugging Face token in platform secrets as `HF_TOKEN` only.
- Never paste a token into a cell, chat, or committed file.
- This notebook never prints the token.

**Before you start**
1. Accept the Gemma license on Hugging Face for `google/gemma-3-1b-it`.
2. **Kaggle:** phone-verify → Session options → GPU T4 · Secrets → `HF_TOKEN` ·
   Add data → attach private Dataset built from `controlsift_kaggle_bundle.zip`.
3. **Colab fallback:** Runtime → GPU · Secrets/`getpass` for `HF_TOKEN` ·
   bootstrap downloads the GitHub Release zip automatically.

Keep `SMOKE = True` for the first session.

**If adapter already exists and you only need the torchao fix — skip Locate:**
```
!pip -q install -U "torchao>=0.16"
```
then run **Install (if needed) → QLoRA eval → Package**. Do not re-run Locate.

**After a successful smoke train + code refresh:** Locate fetches into a temp dir first
(never deletes live WORKDIR until success), preserves `results/`, and falls back to the
Release zip if git fails. Path: **Locate → Install → QLoRA eval → Package**
(skip train if adapter still present).


In [ ]:
# === Config (safe defaults) ===
from pathlib import Path
import os

# Kaggle T4x2: pin one GPU before any torch/CUDA import in later cells.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

SMOKE = True          # True = short validation run; False = full test/challenge + train
SMOKE_LIMIT = 16
REPO_URL = "https://github.com/jtflack-grc/controlsift.git"
REPO_BRANCH = "kaggle-bundle"
KAGGLE_DATASET_DIR = "/kaggle/input"

IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"

print("SMOKE =", SMOKE)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("IN_COLAB =", IN_COLAB)
print("REPO_URL =", REPO_URL, "branch", REPO_BRANCH)
print("WORKDIR =", WORKDIR)


In [ ]:
# === Colab only: fetch or upload the bundle (skip on Kaggle) ===
from pathlib import Path
import zipfile
import shutil
import urllib.request

BUNDLE_URL = "https://github.com/jtflack-grc/controlsift/releases/download/kaggle-bundle/controlsift_kaggle_bundle.zip"

if IN_COLAB:
    dest = Path("/content/controlsift_kaggle_bundle.zip")
    extract_to = Path("/content/_bundle")
    try:
        print("Downloading bundle from GitHub Release…")
        urllib.request.urlretrieve(BUNDLE_URL, dest)
        print("Downloaded", dest, "bytes", dest.stat().st_size)
    except Exception as exc:
        print("Release download failed:", type(exc).__name__, exc)
        from google.colab import files
        print("Upload dist/controlsift_kaggle_bundle.zip from your PC…")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No bundle available")
        name = next(iter(uploaded))
        dest = Path("/content") / name
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    with zipfile.ZipFile(dest, "r") as zf:
        zf.extractall(extract_to)
    candidates = list(extract_to.rglob("pyproject.toml"))
    if not candidates:
        raise SystemExit("Zip missing pyproject.toml")
    src = candidates[0].parent
    work = Path(WORKDIR)
    if work.exists():
        shutil.rmtree(work)
    shutil.copytree(src, work)
    print("Colab bundle ready at", work)
else:
    print("Kaggle session — use attached Dataset (next cell).")


In [ ]:
# === Locate or fetch repo (no secrets) ===
# Safe refresh: never rmtree live WORKDIR until a new tree is ready.
# Fetch order: git --branch → git default+checkout → Release zip.
# Preserves WORKDIR/results (adapter) across refresh. NameError-proof defaults.
import hashlib
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

if "REPO_URL" not in globals():
    REPO_URL = "https://github.com/jtflack-grc/controlsift.git"
if "REPO_BRANCH" not in globals():
    REPO_BRANCH = "kaggle-bundle"
if "KAGGLE_DATASET_DIR" not in globals():
    KAGGLE_DATASET_DIR = "/kaggle/input"
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
if "SMOKE" not in globals():
    SMOKE = True
if "SMOKE_LIMIT" not in globals():
    SMOKE_LIMIT = 16

BUNDLE_URL = (
    "https://github.com/jtflack-grc/controlsift/releases/download/"
    "kaggle-bundle/controlsift_kaggle_bundle.zip"
)

print(
    "Locate defaults:",
    {
        "IN_COLAB": IN_COLAB,
        "WORKDIR": WORKDIR,
        "REPO_URL": REPO_URL,
        "REPO_BRANCH": REPO_BRANCH,
        "SMOKE": SMOKE,
    },
)

work = Path(WORKDIR)
work.parent.mkdir(parents=True, exist_ok=True)
fetch_root = work.parent / "_controlsift_fetch"
PRESERVE_RELPATHS = ("results",)


def _stash_dir() -> Path:
    return work.parent / "_controlsift_preserve"


def stash_workdir_outputs(dest: Path):
    """Copy results/ aside before replacing WORKDIR."""
    if not dest.exists():
        return None
    found = [dest / rel for rel in PRESERVE_RELPATHS if (dest / rel).exists()]
    if not found:
        print("No results/ to preserve (fresh session or train not run yet)")
        return None
    stash = _stash_dir()
    if stash.exists():
        shutil.rmtree(stash)
    stash.mkdir(parents=True)
    for src in found:
        target = stash / src.name
        shutil.copytree(src, target)
        print(f"Preserved {src} -> {target}")
    return stash


def restore_workdir_outputs(dest: Path, stash) -> None:
    """Merge stashed results/ back after successful replace (stashed wins)."""
    if stash is None or not stash.exists():
        return
    for item in stash.iterdir():
        target = dest / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
        print(f"Restored preserved {item.name} -> {target}")
    shutil.rmtree(stash, ignore_errors=True)


def adapter_present(dest: Path) -> bool:
    return (dest / "results" / "gemma_qlora" / "adapter").exists()


def find_bundled_repo(root: Path):
    if not root.exists():
        return None
    for candidate in root.rglob("pyproject.toml"):
        text = candidate.read_text(encoding="utf-8", errors="ignore")
        if 'name = "controlsift"' in text or "name = 'controlsift'" in text:
            return candidate.parent
    return None


def extract_input_zips(root: Path) -> None:
    if not root.exists():
        return
    for zpath in root.rglob("*.zip"):
        dest = Path("/kaggle/working") / "_extracted_dataset" / zpath.stem
        if (dest / "controlsift" / "pyproject.toml").exists() or (dest / "pyproject.toml").exists():
            continue
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting", zpath, "->", dest)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(dest)


def _run_git(cmd, cwd=None):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    print(">>", " ".join(cmd))
    try:
        proc = subprocess.run(
            cmd,
            check=True,
            cwd=cwd,
            env=env,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(
            "git not found on PATH. On Kaggle, Internet must be ON and git available."
        ) from exc
    except subprocess.CalledProcessError as exc:
        print("STDOUT:", (exc.stdout or "")[-2000:])
        print("STDERR:", (exc.stderr or "")[-2000:])
        raise
    if proc.stdout:
        print(proc.stdout[-2000:])
    if proc.stderr:
        print(proc.stderr[-2000:])
    return proc


def _git_head(dest: Path) -> str:
    return subprocess.check_output(
        ["git", "-C", str(dest), "rev-parse", "HEAD"], text=True
    ).strip()


def fetch_via_git_branch(url: str, branch: str, dest: Path) -> str:
    _run_git(["git", "clone", "--depth", "1", "--branch", branch, url, str(dest)])
    sha = _git_head(dest)
    print(f"Git branch clone OK HEAD={sha}")
    return sha


def fetch_via_git_checkout(url: str, branch: str, dest: Path) -> str:
    _run_git(["git", "clone", "--depth", "1", url, str(dest)])
    try:
        _run_git(["git", "fetch", "--depth", "1", "origin", branch], cwd=str(dest))
        _run_git(["git", "checkout", branch], cwd=str(dest))
    except subprocess.CalledProcessError:
        _run_git(["git", "checkout", "-B", branch, f"origin/{branch}"], cwd=str(dest))
    sha = _git_head(dest)
    print(f"Git default+checkout OK HEAD={sha}")
    return sha


def fetch_via_release_zip(dest: Path) -> str:
    zip_path = dest.parent / "_controlsift_bundle.zip"
    extract_to = dest.parent / "_controlsift_zip_extract"
    if zip_path.exists():
        zip_path.unlink()
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    print("Downloading Release zip:", BUNDLE_URL)
    try:
        urllib.request.urlretrieve(BUNDLE_URL, zip_path)
    except Exception as exc:
        raise RuntimeError(
            f"Release zip download failed ({type(exc).__name__}: {exc}). "
            "Confirm Internet ON, or attach Dataset / upload zip."
        ) from exc
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()[:16]
    print(f"Zip bytes={zip_path.stat().st_size} sha256_16={digest}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    src = find_bundled_repo(extract_to)
    if src is None:
        raise RuntimeError("Release zip missing controlsift pyproject.toml")
    shutil.copytree(src, dest)
    print(f"Release zip extract OK digest={digest}")
    return f"zip:{digest}"


def fetch_repo_tree(url: str, branch: str, dest: Path) -> str:
    """Populate dest with a fresh tree. Never touches live WORKDIR."""
    if dest.exists():
        shutil.rmtree(dest)
    errors = []
    for name, fn in (
        ("git clone --branch", lambda: fetch_via_git_branch(url, branch, dest)),
        ("git clone + checkout", lambda: fetch_via_git_checkout(url, branch, dest)),
        ("release zip", lambda: fetch_via_release_zip(dest)),
    ):
        if dest.exists():
            shutil.rmtree(dest)
        try:
            print(f"--- try: {name} ---")
            return fn()
        except Exception as exc:
            msg = f"{name} failed: {type(exc).__name__}: {exc}"
            print(msg)
            errors.append(msg)
            if dest.exists():
                shutil.rmtree(dest, ignore_errors=True)
    raise RuntimeError("All fetch strategies failed.\n" + "\n".join(errors))


def replace_workdir(live: Path, fetched: Path) -> None:
    """Swap live WORKDIR only after fetched tree is ready."""
    backup = live.parent / "_controlsift_old"
    if backup.exists():
        shutil.rmtree(backup)
    if live.exists():
        live.rename(backup)
    try:
        fetched.rename(live)
    except Exception:
        if backup.exists() and not live.exists():
            backup.rename(live)
            print("keeping existing tree (rename failed; restored backup)")
        raise
    if backup.exists():
        shutil.rmtree(backup, ignore_errors=True)


# --- main locate flow ---
ref_id = None
preserved = None
had_existing = work.exists()

try:
    if REPO_URL:
        preserved = stash_workdir_outputs(work)
        ref_id = fetch_repo_tree(REPO_URL, REPO_BRANCH, fetch_root)
        replace_workdir(work, fetch_root)
        restore_workdir_outputs(work, preserved)
        preserved = None
    else:
        extract_input_zips(Path(KAGGLE_DATASET_DIR))
        src = find_bundled_repo(Path(KAGGLE_DATASET_DIR))
        if src is None:
            src = find_bundled_repo(Path("/kaggle/working/_extracted_dataset"))
        if src is not None:
            preserved = stash_workdir_outputs(work)
            if fetch_root.exists():
                shutil.rmtree(fetch_root)
            shutil.copytree(src, fetch_root)
            replace_workdir(work, fetch_root)
            restore_workdir_outputs(work, preserved)
            preserved = None
            ref_id = f"dataset:{src}"
            print("Copied dataset bundle from", src)
        elif Path("/content/controlsift").exists():
            work = Path("/content/controlsift")
            ref_id = "existing:/content/controlsift"
            print("Using existing /content/controlsift")
        else:
            raise RuntimeError(
                "Could not find ControlSift. Set REPO_URL or attach the Dataset zip."
            )
except Exception as exc:
    # Critical: leave existing WORKDIR intact on failure.
    if fetch_root.exists():
        shutil.rmtree(fetch_root, ignore_errors=True)
    if had_existing and work.exists():
        print(
            f"Locate failed ({type(exc).__name__}: {exc}). "
            f"keeping existing tree at {work}"
        )
        print(
            "Adapter present:",
            adapter_present(work),
            work / "results" / "gemma_qlora" / "adapter",
        )
        print(
            "Workaround if you only need torchao: "
            '!pip -q install -U "torchao>=0.16" then run Eval (skip Locate).'
        )
        if (work / "pyproject.toml").exists():
            ref_id = "kept-existing"
        else:
            raise
    else:
        raise

if not work.exists():
    raise RuntimeError(f"WORKDIR missing after locate: {work}")

os.chdir(work)
print("WORKDIR =", work.resolve())
print("REF =", ref_id)
print("Adapter present:", adapter_present(work), work / "results" / "gemma_qlora" / "adapter")
assert (work / "data" / "processed" / "train.jsonl").exists(), "missing train.jsonl"
assert (work / "scripts" / "run_gemma_baseline.py").exists(), "missing scripts"
print("Input tree OK")


In [ ]:
# === Auth via platform secrets only (token never printed) ===
import os
import getpass

def load_hf_token() -> str:
    # Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    # Colab Secrets
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    # Env var fallback (still do not hardcode in a cell)
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    # Last resort on Colab: typed prompt (not stored in the .ipynb source)
    token = getpass.getpass("Paste HF read token (hidden, not saved in notebook): ").strip()
    if token:
        return token
    raise RuntimeError(
        "HF_TOKEN not found. Add it under Kaggle/Colab Secrets, "
        "or paste once at the hidden prompt. Do not put the token in a code cell."
    )

token = load_hf_token()
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token, add_to_git_credential=False)

print("HF login OK; token length =", len(token), "(value not shown)")
del token


In [ ]:
# === Install GPU deps + package ===
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
work = Path(WORKDIR)
if not work.exists():
    raise RuntimeError(f"WORKDIR missing ({work}). Re-run Locate first.")
os.chdir(work)
print("cwd =", Path.cwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError(
        "No GPU visible. Session options \u2192 Accelerator \u2192 GPU T4, then Restart session."
    )

!pip -q install -U pip
# Gemma 3: transformers>=4.50; TRL: SFTConfig; peft: torchao>=0.16 if present.
!pip -q install -U "transformers>=4.50" "trl>=0.14" "torchao>=0.16"
!pip -q install -e ".[gpu]"

# peft>=0.16 raises if stale torchao (e.g. Kaggle 0.10) remains — upgrade then uninstall fallback.
def _torchao_ok() -> bool:
    try:
        import importlib.metadata as md
        ver = md.version("torchao")
    except Exception:
        print("torchao: not installed (ok)")
        return True
    parts = []
    for chunk in ver.split("."):
        digits = "".join(ch for ch in chunk if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
        if len(parts) == 3:
            break
    while len(parts) < 3:
        parts.append(0)
    ok = tuple(parts) >= (0, 16, 0)
    print("torchao:", ver, ("ok" if ok else "TOO OLD"))
    return ok

if not _torchao_ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16"], check=False)
if not _torchao_ok():
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    print("Uninstalled torchao as fallback (peft only probes it)")

import transformers
import trl
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("Install complete")


In [ ]:
# Ensure cwd + smoke flags even if cells are re-run out of order
import os
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
if "SMOKE" not in globals():
    SMOKE = True
if "SMOKE_LIMIT" not in globals():
    SMOKE_LIMIT = 16
work = Path(WORKDIR)
if not work.exists():
    raise RuntimeError(f"WORKDIR missing ({work}). Re-run Locate first.")
os.chdir(work)
print("cwd =", Path.cwd())
print("SMOKE =", SMOKE, "SMOKE_LIMIT =", SMOKE_LIMIT)

# === Gemma baselines ===
import subprocess
import sys

splits = ["validation"] if SMOKE else ["validation", "test", "challenge"]

for mode in ["zero_shot", "few_shot"]:
    for split in splits:
        cmd = [
            sys.executable,
            "scripts/run_gemma_baseline.py",
            "--mode",
            mode,
            "--split",
            split,
        ]
        if SMOKE:
            cmd.extend(["--limit", str(SMOKE_LIMIT)])
        print("\n>>", " ".join(cmd))
        subprocess.run(cmd, check=True)
print("Baselines complete")


In [ ]:
# Ensure cwd + smoke flags even if cells are re-run out of order
import os
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
if "SMOKE" not in globals():
    SMOKE = True
if "SMOKE_LIMIT" not in globals():
    SMOKE_LIMIT = 16
work = Path(WORKDIR)
if not work.exists():
    raise RuntimeError(f"WORKDIR missing ({work}). Re-run Locate first.")
os.chdir(work)
print("cwd =", Path.cwd())
print("SMOKE =", SMOKE, "SMOKE_LIMIT =", SMOKE_LIMIT)

# === QLoRA train ===
import subprocess
import sys

train_cmd = [
    sys.executable,
    "-m",
    "controlsift.training.train",
]
if SMOKE:
    train_cmd.append("--smoke")
print(">>", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)
print("Train complete — expect precision print with fp16_amp: False on T4")


In [ ]:
# Ensure cwd + smoke flags even if cells are re-run out of order
import os
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
if "SMOKE" not in globals():
    SMOKE = True
if "SMOKE_LIMIT" not in globals():
    SMOKE_LIMIT = 16
work = Path(WORKDIR)
if not work.exists():
    raise RuntimeError(f"WORKDIR missing ({work}). Re-run Locate first.")
os.chdir(work)
print("cwd =", Path.cwd())
print("SMOKE =", SMOKE, "SMOKE_LIMIT =", SMOKE_LIMIT)

# === QLoRA eval ===
import subprocess
import sys
from pathlib import Path

adapter = Path("results/gemma_qlora/adapter")
if not adapter.exists():
    raise RuntimeError(
        f"Adapter missing at {adapter.resolve()}. Re-run train cell first."
    )

limit_flag = ["--limit", str(SMOKE_LIMIT)] if SMOKE else []
splits = ["validation"] if SMOKE else ["validation", "test", "challenge"]

for split in splits:
    cmd = [
        sys.executable,
        "scripts/run_gemma_qlora_eval.py",
        "--split",
        split,
        *limit_flag,
    ]
    print("\n>>", " ".join(cmd))
    subprocess.run(cmd, check=True)
print("QLoRA eval complete")


In [ ]:
# Ensure cwd + smoke flags even if cells are re-run out of order
import os
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
if "IN_COLAB" not in globals():
    IN_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
if "WORKDIR" not in globals():
    WORKDIR = "/content/controlsift" if IN_COLAB else "/kaggle/working/controlsift"
if "SMOKE" not in globals():
    SMOKE = True
if "SMOKE_LIMIT" not in globals():
    SMOKE_LIMIT = 16
work = Path(WORKDIR)
if not work.exists():
    raise RuntimeError(f"WORKDIR missing ({work}). Re-run Locate first.")
os.chdir(work)
print("cwd =", Path.cwd())
print("SMOKE =", SMOKE, "SMOKE_LIMIT =", SMOKE_LIMIT)

# === Package downloadable outputs (metrics/preds/meta; skip huge weights by default) ===
import json
import zipfile
from pathlib import Path

INCLUDE_LARGE_ADAPTER = False  # set True only if you knowingly want a large zip

out_zip = Path("/kaggle/working/controlsift_gpu_outputs.zip")
if not out_zip.parent.exists():
    out_zip = Path("/content/controlsift_gpu_outputs.zip") if IN_COLAB else Path("controlsift_gpu_outputs.zip")

roots = [
    Path("results/gemma_zero_shot"),
    Path("results/gemma_few_shot"),
    Path("results/gemma_qlora"),
]

written = []
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            rel = str(path).replace("\\", "/")
            if path.suffix in {".safetensors", ".bin", ".pt", ".pth"} and not INCLUDE_LARGE_ADAPTER:
                continue
            if "adapter/" in rel and path.suffix not in {".json", ".txt", ".md"} and not INCLUDE_LARGE_ADAPTER:
                # keep tokenizer/config json; skip weight shards
                if path.name.startswith("adapter_model") or path.name.startswith("model"):
                    continue
            zf.write(path, arcname=rel)
            written.append(rel)

print("Wrote", out_zip.resolve(), "files:", len(written))
print("Download this zip → unzip into your local repo root → run:")
print("  python scripts/after_kaggle.py --zip path/to/controlsift_gpu_outputs.zip")

def _macro(path: Path):
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("metrics", {}).get("macro_f1")
    except Exception:
        return None

status = {}
for exp in ("gemma_zero_shot", "gemma_few_shot", "gemma_qlora"):
    status[exp] = {
        "validation_macro_f1": _macro(Path("results") / exp / "metrics_validation.json"),
        "test_macro_f1": _macro(Path("results") / exp / "metrics_test.json"),
    }
print(json.dumps({"packaged_metrics": status, "SMOKE": SMOKE}, indent=2))
if SMOKE:
    print("NOTE: smoke writes validation only — set SMOKE=False for public test metrics")


## After download (on your PC)

1. Unzip into the ControlSift repo root (so `results/gemma_*` merge in).
2. `python scripts/after_kaggle.py`
3. Refresh the local site — Gemma rows leave `pending` only after **full** (`SMOKE = False`) writes `metrics_test.json`.
4. If this was smoke only, re-run on Kaggle with `SMOKE = False` when you have quota, then repeat steps 1–3.
5. Push `results/` + `docs/` to GitHub so Pages publishes (workflow already configured).

Do not commit HF tokens. Prefer committing metrics/predictions JSON only.